# CogAttention — Vigilance Probe

**Track:** Attention — Sustained Attention
**Benchmark:** CogAttention v1.0
**Task:** sustained

---

## Methodology

Tests sustained attention through vigilance probes — detecting targets scattered across long documents with near-miss distractors. Measures whether detection drops off later in the document (vigilance decrement). Based on CPT (Mackworth, 1948).

### Cognitive Science Grounding

- **Continuous Performance Test / Vigilance** (Mackworth, 1948): detection accuracy degrades over time even when targets remain constant
- In LLMs, this manifests as the Lost-in-the-Middle phenomenon — softmax attention dilution and RoPE decay reduce mid-document target detection

### Difficulty Scaling

Easy: 10 paragraphs | Medium: 25 paragraphs | Hard: 50 paragraphs | Expert: 80 paragraphs | Frontier: 150+ paragraphs with subtler targets

### Scoring

One assertion per target item — checks whether each scattered target was found. Measures vigilance decrement by position.

All instances are procedurally generated from a seed with programmatic ground truth.
No static datasets. 7 layers of contamination resistance including canary strings,
zero lexical overlap (Semantic NIAH), and seed-based regeneration.

---

`<!-- COGATTENTION-BENCH-CANARY-D39C5949C6D1 -->`


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 2: Imports + Inline Helpers
# CogAttention — Sustained Attention
# ══════════════════════════════════════════════════════════════════════

import kaggle_benchmarks as kbench

import json
import re

def extract_answer_block(response):
    for pat in [r"ANSWER:\s*(.*)", r"Answer:\s*(.*)", r"answer:\s*(.*)"]:
        match = re.search(pat, response, re.DOTALL | re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return response.strip()

def extract_numbered_answers(response):
    answer_block = extract_answer_block(response)
    results = {}
    matches = re.findall(
        r"(\d+)\s*[.):\-]\s*(.+?)(?=\n\d+\s*[.):\-]|\Z)",
        answer_block, re.DOTALL,
    )
    for num, val in matches:
        results[num] = val.strip().rstrip(".")
    return results

def extract_list_items(response):
    answer_block = extract_answer_block(response)
    bullets = re.findall(r"[-\u2022]\s*(.+?)(?:\n|$)", answer_block)
    if bullets:
        return [b.strip().rstrip(".") for b in bullets]
    numeric_items = re.findall(
        r'[\$]?\d{1,3}(?:,\d{3})*(?:\.\d+)?(?:\s*(?:\xb0[CF]|mg/L|%|\$))?',
        answer_block,
    )
    if numeric_items and len(numeric_items) >= 2:
        return [x.strip() for x in numeric_items]
    if "," in answer_block:
        items = [x.strip().rstrip(".") for x in answer_block.split(",")]
        return [x for x in items if x]
    lines = [l.strip().rstrip(".") for l in answer_block.split("\n") if l.strip()]
    return lines if lines else ([answer_block] if answer_block else [])

def extract_person_item_pairs(response):
    answer_block = extract_answer_block(response)
    results = {}
    for pat in [
        r"[-\u2022]?\s*(\w+)\s*:\s*(.+?)(?:\n|$)",
        r"[-\u2022]?\s*(\w+)\s+holds?\s+(?:a\s+)?(.+?)(?:\n|$)",
    ]:
        matches = re.findall(pat, answer_block, re.IGNORECASE)
        if matches:
            for name, item in matches:
                results[name.strip()] = item.strip().rstrip(".")
            break
    return results

def fuzzy_value_match(predicted, gold):
    pred_clean = re.sub(r"\s+", " ", predicted.strip().lower())
    gold_clean = re.sub(r"\s+", " ", gold.strip().lower())
    if pred_clean == gold_clean:
        return True
    if gold_clean in pred_clean:
        return True
    try:
        pred_num = float(re.sub(r"[,$%\xb0]", "", predicted))
        gold_num = float(re.sub(r"[,$%\xb0]", "", gold))
        return pred_num == gold_num
    except (ValueError, TypeError):
        pass
    return False

def _escape_for_regex(s):
    return re.escape(s).replace(r"\ ", r"\s+")


def run_assertions_sustained(response, gold, kbench):
    for target in gold["targets"]:
        pattern = rf"(?i){_escape_for_regex(target)}"
        kbench.assertions.assert_contains_regex(
            pattern, response,
            expectation=f"Should find target '{target}'"
        )


print("CogAttention helpers loaded")
print(f"Task types: ['sustained']")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 3: Task Definitions + Embedded Dataset
# ══════════════════════════════════════════════════════════════════════


@kbench.task(name="cogattention_sustained")
def cogattention_sustained(llm, prompt: str, gold_json: str, task_id: str, difficulty: str):
    """CogAttention sustained task."""
    response = llm.prompt(prompt)
    gold = json.loads(gold_json)
    run_assertions_sustained(response, gold, kbench)


# ── Embedded dataset ──────────────────────────────────────────────────
DATASET = json.loads(r'''
[
 {
  "task_id": "sustained_easy_000",
  "task_type": "sustained",
  "difficulty": "Easy",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a type of bird. List every bird you find.\n\nImportant: There may be similar-sounding items that are NOT a type of bird — do not include those.\n\n---\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. The logbook recorded a sugar glider at the northern edge of the district.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Uma mentioned seeing a pterodactyl while crossing the square.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A toucan was noted in the margin of the inspector's report.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient. A flamingo was noted in the margin of the inspector's report.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Reports from the harbour mentioned a raven had been observed twice that week.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Idris reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Joaquin fully understood. The wooden shelves bowed slightly under the weight. A ibis was noted in the margin of the inspector's report.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A quail was spotted near the old bridge that morning.\n---\n\nList ALL birds mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"toucan\", \"flamingo\", \"raven\", \"ibis\", \"quail\"], \"nearmisses\": [\"sugar glider\", \"pterodactyl\"]}"
 },
 {
  "task_id": "sustained_easy_001",
  "task_type": "sustained",
  "difficulty": "Easy",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river — do not include those.\n\n---\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Orla recalled that a Euphrates had appeared briefly near the market.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Among the items catalogued was a Lake Victoria, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient. A Mekong was spotted near the old bridge that morning.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The survey team documented a Panama Canal in the area surrounding Bruges.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. A Congo was noted in the margin of the inspector's report.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. A Indus was noted in the margin of the inspector's report.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. A Rhine was noted in the margin of the inspector's report.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Euphrates\", \"Mekong\", \"Congo\", \"Indus\", \"Rhine\"], \"nearmisses\": [\"Lake Victoria\", \"Panama Canal\"]}"
 },
 {
  "task_id": "sustained_easy_002",
  "task_type": "sustained",
  "difficulty": "Easy",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a metallic element. List every metal you find.\n\nImportant: There may be similar-sounding items that are NOT a metallic element — do not include those.\n\n---\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Among the items catalogued was a cobalt, noted without further comment.\n\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The logbook recorded a platinum at the northern edge of the district.\n\nA thin rain began to fall just as Nalini reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Viktor Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Joelle mentioned seeing a titanium while crossing the square.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The logbook recorded a chalk at the northern edge of the district.\n\nThe old postal route between Oulu and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Reports from the harbour mentioned a ceramic had been observed twice that week.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yara fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a osmium had been observed twice that week.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The logbook recorded a palladium at the northern edge of the district.\n---\n\nList ALL metals mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"cobalt\", \"platinum\", \"titanium\", \"osmium\", \"palladium\"], \"nearmisses\": [\"chalk\", \"ceramic\"]}"
 },
 {
  "task_id": "sustained_easy_003",
  "task_type": "sustained",
  "difficulty": "Easy",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument — do not include those.\n\n---\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Elio fully understood. The wooden shelves bowed slightly under the weight.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Orla mentioned seeing a sitar while crossing the square.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The logbook recorded a metronome at the northern edge of the district.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient. Sigrid recalled that a harp had appeared briefly near the market.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. A balalaika was noted in the margin of the inspector's report.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A microphone was noted in the margin of the inspector's report.\n\nA thin rain began to fall just as Ravi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Qadir Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A mandolin was noted in the margin of the inspector's report.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Among the items catalogued was a violin, noted without further comment.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"sitar\", \"harp\", \"balalaika\", \"mandolin\", \"violin\"], \"nearmisses\": [\"metronome\", \"microphone\"]}"
 },
 {
  "task_id": "sustained_easy_004",
  "task_type": "sustained",
  "difficulty": "Easy",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a type of bird. List every bird you find.\n\nImportant: There may be similar-sounding items that are NOT a type of bird — do not include those.\n\n---\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Celine fully understood. The wooden shelves bowed slightly under the weight. Among the items catalogued was a dove, noted without further comment.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Haruto fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Yara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Among the items catalogued was a kingfisher, noted without further comment.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A wasp was spotted near the old bridge that morning.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Colette mentioned seeing a pelican while crossing the square.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The logbook recorded a parrot at the northern edge of the district.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Priya fully understood. The wooden shelves bowed slightly under the weight. Olena mentioned seeing a flying squirrel while crossing the square.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Vesna recalled that a flamingo had appeared briefly near the market.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n---\n\nList ALL birds mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"dove\", \"kingfisher\", \"pelican\", \"parrot\", \"flamingo\"], \"nearmisses\": [\"wasp\", \"flying squirrel\"]}"
 },
 {
  "task_id": "sustained_easy_005",
  "task_type": "sustained",
  "difficulty": "Easy",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument — do not include those.\n\n---\nThe market square in Cusco was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Sigrid mentioned seeing a tabla while crossing the square.\n\nA thin rain began to fall just as Zain reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The survey team documented a erhu in the area surrounding Gdansk.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Ravi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Reports from the harbour mentioned a lute had been observed twice that week.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tariq fully understood. The wooden shelves bowed slightly under the weight. A microphone was spotted near the old bridge that morning.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The logbook recorded a oud at the northern edge of the district.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient. Reports from the harbour mentioned a headphones had been observed twice that week.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The survey team documented a balalaika in the area surrounding Mandalay.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"tabla\", \"erhu\", \"lute\", \"oud\", \"balalaika\"], \"nearmisses\": [\"microphone\", \"headphones\"]}"
 },
 {
  "task_id": "sustained_easy_006",
  "task_type": "sustained",
  "difficulty": "Easy",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a metallic element. List every metal you find.\n\nImportant: There may be similar-sounding items that are NOT a metallic element — do not include those.\n\n---\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Kaia mentioned seeing a osmium while crossing the square.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Zain reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Reports from the harbour mentioned a lead had been observed twice that week.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. A ruthenium was noted in the margin of the inspector's report.\n\nThe workshop on Bram Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Nico mentioned seeing a chalk while crossing the square.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. A palladium was spotted near the old bridge that morning.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The survey team documented a platinum in the area surrounding Ulaanbaatar.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Magnus recalled that a concrete had appeared briefly near the market.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n---\n\nList ALL metals mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"osmium\", \"lead\", \"ruthenium\", \"palladium\", \"platinum\"], \"nearmisses\": [\"chalk\", \"concrete\"]}"
 },
 {
  "task_id": "sustained_easy_007",
  "task_type": "sustained",
  "difficulty": "Easy",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument — do not include those.\n\n---\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Sigrid fully understood. The wooden shelves bowed slightly under the weight. Sigrid mentioned seeing a theremin while crossing the square.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Bashir recalled that a music stand had appeared briefly near the market.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Reports from the harbour mentioned a pitch pipe had been observed twice that week.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Zain reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A flute was noted in the margin of the inspector's report.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Lumi fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a dulcimer had been observed twice that week.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Zanzibar insisted on maintaining their position, while the merchants grew impatient. Ravi recalled that a harp had appeared briefly near the market.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The survey team documented a violin in the area surrounding Recife.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"theremin\", \"flute\", \"dulcimer\", \"harp\", \"violin\"], \"nearmisses\": [\"music stand\", \"pitch pipe\"]}"
 },
 {
  "task_id": "sustained_medium_008",
  "task_type": "sustained",
  "difficulty": "Medium",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river — do not include those.\n\n---\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Trieste insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Bashir Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Amara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The logbook recorded a Congo at the northern edge of the district.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Reports from the harbour mentioned a Lake Baikal had been observed twice that week.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Among the items catalogued was a Don, noted without further comment.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Gael mentioned seeing a Bay of Bengal while crossing the square.\n\nA thin rain began to fall just as Amara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The survey team documented a Panama Canal in the area surrounding Kumasi.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Wren fully understood. The wooden shelves bowed slightly under the weight.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Kaia mentioned seeing a Elbe while crossing the square.\n\nThe workshop on Yara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Reports from the harbour mentioned a Yangtze had been observed twice that week.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Kenji recalled that a Mekong had appeared briefly near the market.\n\nA thin rain began to fall just as Adaeze reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe market square in Tallinn was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The survey team documented a Danube in the area surrounding Kumasi.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Ines mentioned seeing a Rhine while crossing the square.\n\nA thin rain began to fall just as Orla reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Reports from the harbour mentioned a Dead Sea had been observed twice that week.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Magnus recalled that a Aral Sea had appeared briefly near the market.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. A Volga was noted in the margin of the inspector's report.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Congo\", \"Don\", \"Elbe\", \"Yangtze\", \"Mekong\", \"Danube\", \"Rhine\", \"Volga\"], \"nearmisses\": [\"Lake Baikal\", \"Bay of Bengal\", \"Panama Canal\", \"Dead Sea\", \"Aral Sea\"]}"
 },
 {
  "task_id": "sustained_medium_009",
  "task_type": "sustained",
  "difficulty": "Medium",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a metallic element. List every metal you find.\n\nImportant: There may be similar-sounding items that are NOT a metallic element — do not include those.\n\n---\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Reports from the harbour mentioned a vanadium had been observed twice that week.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. A granite was spotted near the old bridge that morning.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. A rubber was noted in the margin of the inspector's report.\n\nA thin rain began to fall just as Yara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A copper was noted in the margin of the inspector's report.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A palladium was noted in the margin of the inspector's report.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The logbook recorded a zinc at the northern edge of the district.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Reports from the harbour mentioned a osmium had been observed twice that week.\n\nThe workshop on Ugo Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A tin was spotted near the old bridge that morning.\n\nA thin rain began to fall just as Orla reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Among the items catalogued was a glass, noted without further comment.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The logbook recorded a ceramic at the northern edge of the district.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Reports from the harbour mentioned a iridium had been observed twice that week.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Zora recalled that a rhodium had appeared briefly near the market.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Uma recalled that a sand had appeared briefly near the market.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n---\n\nList ALL metals mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"vanadium\", \"copper\", \"palladium\", \"zinc\", \"osmium\", \"tin\", \"iridium\", \"rhodium\"], \"nearmisses\": [\"granite\", \"rubber\", \"glass\", \"ceramic\", \"sand\"]}"
 },
 {
  "task_id": "sustained_medium_010",
  "task_type": "sustained",
  "difficulty": "Medium",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river — do not include those.\n\n---\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. A Rhine was noted in the margin of the inspector's report.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Jaipur insisted on maintaining their position, while the merchants grew impatient.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Reports from the harbour mentioned a Panama Canal had been observed twice that week.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient. Kenji mentioned seeing a Murray while crossing the square.\n\nThe workshop on Greta Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The logbook recorded a Danube at the northern edge of the district.\n\nA thin rain began to fall just as Zain reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Dmitri fully understood. The wooden shelves bowed slightly under the weight. Celine mentioned seeing a Lake Victoria while crossing the square.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Maren fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Ravi fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kumasi insisted on maintaining their position, while the merchants grew impatient. Dariush mentioned seeing a Mississippi while crossing the square.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Reports from the harbour mentioned a Volga had been observed twice that week.\n\nThe old postal route between Kotor and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The survey team documented a Bay of Bengal in the area surrounding Recife.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The survey team documented a Aral Sea in the area surrounding Fez.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. A Dead Sea was spotted near the old bridge that morning.\n\nThe old postal route between Oulu and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Gael mentioned seeing a Oder while crossing the square.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The survey team documented a Euphrates in the area surrounding Fez.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient. A Indus was noted in the margin of the inspector's report.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Rhine\", \"Murray\", \"Danube\", \"Mississippi\", \"Volga\", \"Oder\", \"Euphrates\", \"Indus\"], \"nearmisses\": [\"Panama Canal\", \"Lake Victoria\", \"Bay of Bengal\", \"Aral Sea\", \"Dead Sea\"]}"
 },
 {
  "task_id": "sustained_medium_011",
  "task_type": "sustained",
  "difficulty": "Medium",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a type of bird. List every bird you find.\n\nImportant: There may be similar-sounding items that are NOT a type of bird — do not include those.\n\n---\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Femi mentioned seeing a finch while crossing the square.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Olena Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. A pelican was noted in the margin of the inspector's report.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Tariq recalled that a oriole had appeared briefly near the market.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Luang Prabang insisted on maintaining their position, while the merchants grew impatient. The logbook recorded a dove at the northern edge of the district.\n\nA thin rain began to fall just as Dariush reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Greta mentioned seeing a heron while crossing the square.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA thin rain began to fall just as Yuki reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Among the items catalogued was a moth, noted without further comment.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The logbook recorded a quail at the northern edge of the district.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Among the items catalogued was a osprey, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient. Willa recalled that a bat had appeared briefly near the market.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Reports from the harbour mentioned a wasp had been observed twice that week.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Sigrid Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yara fully understood. The wooden shelves bowed slightly under the weight.\n\nThe workshop on Xander Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The logbook recorded a butterfly at the northern edge of the district.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Gael fully understood. The wooden shelves bowed slightly under the weight. The survey team documented a flying fish in the area surrounding Kotor.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kotor insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Among the items catalogued was a raven, noted without further comment.\n---\n\nList ALL birds mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"finch\", \"pelican\", \"oriole\", \"dove\", \"heron\", \"quail\", \"osprey\", \"raven\"], \"nearmisses\": [\"moth\", \"bat\", \"wasp\", \"butterfly\", \"flying fish\"]}"
 },
 {
  "task_id": "sustained_medium_012",
  "task_type": "sustained",
  "difficulty": "Medium",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a type of bird. List every bird you find.\n\nImportant: There may be similar-sounding items that are NOT a type of bird — do not include those.\n\n---\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. A flying squirrel was noted in the margin of the inspector's report.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The logbook recorded a raven at the northern edge of the district.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. A moth was spotted near the old bridge that morning.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Reports from the harbour mentioned a oriole had been observed twice that week.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kumasi insisted on maintaining their position, while the merchants grew impatient. The survey team documented a quail in the area surrounding Recife.\n\nThe market square in Valetta was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Among the items catalogued was a wasp, noted without further comment.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. A woodpecker was noted in the margin of the inspector's report.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The survey team documented a sparrow in the area surrounding Tbilisi.\n\nThe workshop on Soren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Among the items catalogued was a puffin, noted without further comment.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Among the items catalogued was a magpie, noted without further comment.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Olena mentioned seeing a pterodactyl while crossing the square.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The survey team documented a osprey in the area surrounding Gdansk.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The logbook recorded a flying fish at the northern edge of the district.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Olena fully understood. The wooden shelves bowed slightly under the weight.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n---\n\nList ALL birds mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"raven\", \"oriole\", \"quail\", \"woodpecker\", \"sparrow\", \"puffin\", \"magpie\", \"osprey\"], \"nearmisses\": [\"flying squirrel\", \"moth\", \"wasp\", \"pterodactyl\", \"flying fish\"]}"
 },
 {
  "task_id": "sustained_medium_013",
  "task_type": "sustained",
  "difficulty": "Medium",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument — do not include those.\n\n---\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Among the items catalogued was a dulcimer, noted without further comment.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Ines fully understood. The wooden shelves bowed slightly under the weight.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Tala reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient. The logbook recorded a metronome at the northern edge of the district.\n\nThe workshop on Sigrid Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. A tuning fork was spotted near the old bridge that morning.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A amplifier was noted in the margin of the inspector's report.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Yuki recalled that a bassoon had appeared briefly near the market.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Reports from the harbour mentioned a harp had been observed twice that week.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A mbira was spotted near the old bridge that morning.\n\nA thin rain began to fall just as Zora reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Leif mentioned seeing a hurdy-gurdy while crossing the square.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The survey team documented a microphone in the area surrounding Reykjavik.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A erhu was spotted near the old bridge that morning.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Nico recalled that a cello had appeared briefly near the market.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cusco insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Reports from the harbour mentioned a mixer had been observed twice that week.\n\nThe workshop on Tariq Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Freya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A balalaika was spotted near the old bridge that morning.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"dulcimer\", \"bassoon\", \"harp\", \"mbira\", \"hurdy-gurdy\", \"erhu\", \"cello\", \"balalaika\"], \"nearmisses\": [\"metronome\", \"tuning fork\", \"amplifier\", \"microphone\", \"mixer\"]}"
 },
 {
  "task_id": "sustained_medium_014",
  "task_type": "sustained",
  "difficulty": "Medium",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument — do not include those.\n\n---\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Jaipur and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Soren fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Ravi mentioned seeing a cello while crossing the square.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Reports from the harbour mentioned a sitar had been observed twice that week.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Freya fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The survey team documented a music stand in the area surrounding Luang Prabang.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cartagena insisted on maintaining their position, while the merchants grew impatient. Reports from the harbour mentioned a amplifier had been observed twice that week.\n\nA thin rain began to fall just as Tala reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Hana mentioned seeing a oud while crossing the square.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Reports from the harbour mentioned a mbira had been observed twice that week.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Magnus recalled that a tabla had appeared briefly near the market.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The logbook recorded a dulcimer at the northern edge of the district.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Among the items catalogued was a timpani, noted without further comment.\n\nThe workshop on Nalini Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The survey team documented a headphones in the area surrounding Trieste.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The logbook recorded a mandolin at the northern edge of the district.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Among the items catalogued was a record player, noted without further comment.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Sigrid reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A pitch pipe was noted in the margin of the inspector's report.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"cello\", \"sitar\", \"oud\", \"mbira\", \"tabla\", \"dulcimer\", \"timpani\", \"mandolin\"], \"nearmisses\": [\"music stand\", \"amplifier\", \"headphones\", \"record player\", \"pitch pipe\"]}"
 },
 {
  "task_id": "sustained_medium_015",
  "task_type": "sustained",
  "difficulty": "Medium",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river — do not include those.\n\n---\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient. Among the items catalogued was a Dead Sea, noted without further comment.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Nalini recalled that a Ganges had appeared briefly near the market.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Reports from the harbour mentioned a Danube had been observed twice that week.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The survey team documented a Strait of Gibraltar in the area surrounding Ulaanbaatar.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Olena mentioned seeing a Caspian Sea while crossing the square.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Oulu insisted on maintaining their position, while the merchants grew impatient.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Wren fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a Black Sea had been observed twice that week.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A Bay of Bengal was noted in the margin of the inspector's report.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Kotor and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. A Amazon was noted in the margin of the inspector's report.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Reports from the harbour mentioned a Mekong had been observed twice that week.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Among the items catalogued was a Don, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kumasi insisted on maintaining their position, while the merchants grew impatient. Reports from the harbour mentioned a Mississippi had been observed twice that week.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Wren fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Ugo fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a Murray had been observed twice that week.\n\nThe workshop on Nico Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Elara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Among the items catalogued was a Zambezi, noted without further comment.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Ganges\", \"Danube\", \"Amazon\", \"Mekong\", \"Don\", \"Mississippi\", \"Murray\", \"Zambezi\"], \"nearmisses\": [\"Dead Sea\", \"Strait of Gibraltar\", \"Caspian Sea\", \"Black Sea\", \"Bay of Bengal\"]}"
 },
 {
  "task_id": "sustained_hard_016",
  "task_type": "sustained",
  "difficulty": "Hard",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument — do not include those.\n\n---\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The survey team documented a erhu in the area surrounding Gdansk.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Reports from the harbour mentioned a amplifier had been observed twice that week.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Among the items catalogued was a microphone, noted without further comment.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. A koto was noted in the margin of the inspector's report.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Adaeze Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Luang Prabang insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Maren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Among the items catalogued was a metronome, noted without further comment.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A pitch pipe was noted in the margin of the inspector's report.\n\nA thin rain began to fall just as Freya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe workshop on Soren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Paloma fully understood. The wooden shelves bowed slightly under the weight. The survey team documented a headphones in the area surrounding Luang Prabang.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The survey team documented a harp in the area surrounding Trieste.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Yara mentioned seeing a hurdy-gurdy while crossing the square.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Wren fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a record player had been observed twice that week.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient. The survey team documented a timpani in the area surrounding Ulaanbaatar.\n\nA thin rain began to fall just as Idris reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Paloma mentioned seeing a mbira while crossing the square.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A balalaika was noted in the margin of the inspector's report.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Valetta was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Willa fully understood. The wooden shelves bowed slightly under the weight. A mixer was noted in the margin of the inspector's report.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Paloma fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The logbook recorded a theremin at the northern edge of the district.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Uma fully understood. The wooden shelves bowed slightly under the weight. The logbook recorded a music stand at the northern edge of the district.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Idris Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Priya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Elio mentioned seeing a tabla while crossing the square.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Zanzibar insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient. Reports from the harbour mentioned a sitar had been observed twice that week.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"erhu\", \"koto\", \"harp\", \"hurdy-gurdy\", \"timpani\", \"mbira\", \"balalaika\", \"theremin\", \"tabla\", \"sitar\"], \"nearmisses\": [\"amplifier\", \"microphone\", \"metronome\", \"pitch pipe\", \"headphones\", \"record player\", \"mixer\", \"music stand\"]}"
 },
 {
  "task_id": "sustained_hard_017",
  "task_type": "sustained",
  "difficulty": "Hard",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river — do not include those.\n\n---\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Hana reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Elio reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A Tigris was spotted near the old bridge that morning.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A Panama Canal was spotted near the old bridge that morning.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Among the items catalogued was a Volga, noted without further comment.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe workshop on Sigrid Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Among the items catalogued was a Dead Sea, noted without further comment.\n\nA thin rain began to fall just as Lumi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The survey team documented a Lake Victoria in the area surrounding Kumasi.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Reports from the harbour mentioned a Congo had been observed twice that week.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. A Lake Baikal was spotted near the old bridge that morning.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Reports from the harbour mentioned a Strait of Gibraltar had been observed twice that week.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Kaia mentioned seeing a Oder while crossing the square.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Among the items catalogued was a Rhine, noted without further comment.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Zora mentioned seeing a Indus while crossing the square.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The survey team documented a Bay of Bengal in the area surrounding Ulaanbaatar.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. The logbook recorded a Suez Canal at the northern edge of the district.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The survey team documented a Mekong in the area surrounding Fez.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Bram recalled that a Nile had appeared briefly near the market.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Ugo fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Bashir mentioned seeing a Aral Sea while crossing the square.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Idris mentioned seeing a Tagus while crossing the square.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Kotor and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Reports from the harbour mentioned a Elbe had been observed twice that week.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Tigris\", \"Volga\", \"Congo\", \"Oder\", \"Rhine\", \"Indus\", \"Mekong\", \"Nile\", \"Tagus\", \"Elbe\"], \"nearmisses\": [\"Panama Canal\", \"Dead Sea\", \"Lake Victoria\", \"Lake Baikal\", \"Strait of Gibraltar\", \"Bay of Bengal\", \"Suez Canal\", \"Aral Sea\"]}"
 },
 {
  "task_id": "sustained_hard_018",
  "task_type": "sustained",
  "difficulty": "Hard",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river — do not include those.\n\n---\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Ines reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Maren recalled that a Indus had appeared briefly near the market.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Nalini Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe workshop on Zora Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Olena reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A Dead Sea was spotted near the old bridge that morning.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. A Congo was noted in the margin of the inspector's report.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The survey team documented a Murray in the area surrounding Tbilisi.\n\nA thin rain began to fall just as Colette reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Kenji Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Hana fully understood. The wooden shelves bowed slightly under the weight. The logbook recorded a Aral Sea at the northern edge of the district.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA thin rain began to fall just as Wren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Bram reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Xander Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Elara recalled that a Ganges had appeared briefly near the market.\n\nThe market square in Tallinn was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. A Lake Victoria was spotted near the old bridge that morning.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cartagena insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Zain recalled that a Panama Canal had appeared briefly near the market.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Femi recalled that a Tigris had appeared briefly near the market.\n\nA thin rain began to fall just as Elio reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The logbook recorded a Mekong at the northern edge of the district.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. A Elbe was noted in the margin of the inspector's report.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Olena mentioned seeing a Yangtze while crossing the square.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yuki fully understood. The wooden shelves bowed slightly under the weight. A Amazon was spotted near the old bridge that morning.\n\nA thin rain began to fall just as Hana reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Sigrid mentioned seeing a Bay of Bengal while crossing the square.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The logbook recorded a Suez Canal at the northern edge of the district.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Among the items catalogued was a Mississippi, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient. The logbook recorded a Caspian Sea at the northern edge of the district.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Haruto Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Reports from the harbour mentioned a Strait of Gibraltar had been observed twice that week.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Qadir fully understood. The wooden shelves bowed slightly under the weight.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Indus\", \"Congo\", \"Murray\", \"Ganges\", \"Tigris\", \"Mekong\", \"Elbe\", \"Yangtze\", \"Amazon\", \"Mississippi\"], \"nearmisses\": [\"Dead Sea\", \"Aral Sea\", \"Lake Victoria\", \"Panama Canal\", \"Bay of Bengal\", \"Suez Canal\", \"Caspian Sea\", \"Strait of Gibraltar\"]}"
 },
 {
  "task_id": "sustained_hard_019",
  "task_type": "sustained",
  "difficulty": "Hard",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a metallic element. List every metal you find.\n\nImportant: There may be similar-sounding items that are NOT a metallic element — do not include those.\n\n---\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cartagena insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe workshop on Adaeze Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The survey team documented a granite in the area surrounding Cusco.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Reports from the harbour mentioned a plastic had been observed twice that week.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Ugo fully understood. The wooden shelves bowed slightly under the weight. A lead was noted in the margin of the inspector's report.\n\nThe workshop on Ravi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Reports from the harbour mentioned a ceramic had been observed twice that week.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The logbook recorded a niobium at the northern edge of the district.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Dmitri reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Nico fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Kotor and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Freya recalled that a chalk had appeared briefly near the market.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A zinc was spotted near the old bridge that morning.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The logbook recorded a rubber at the northern edge of the district.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA thin rain began to fall just as Xander reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A titanium was spotted near the old bridge that morning.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Amara fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Zora reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Adaeze fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Soren mentioned seeing a chromium while crossing the square.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The survey team documented a iron in the area surrounding Tallinn.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Cusco and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Among the items catalogued was a tungsten, noted without further comment.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The survey team documented a copper in the area surrounding Jaipur.\n\nThe workshop on Tala Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA thin rain began to fall just as Olena reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The logbook recorded a glass at the northern edge of the district.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Magnus recalled that a sand had appeared briefly near the market.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Among the items catalogued was a wood, noted without further comment.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Reports from the harbour mentioned a osmium had been observed twice that week.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A vanadium was noted in the margin of the inspector's report.\n---\n\nList ALL metals mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"lead\", \"niobium\", \"zinc\", \"titanium\", \"chromium\", \"iron\", \"tungsten\", \"copper\", \"osmium\", \"vanadium\"], \"nearmisses\": [\"granite\", \"plastic\", \"ceramic\", \"chalk\", \"rubber\", \"glass\", \"sand\", \"wood\"]}"
 },
 {
  "task_id": "sustained_hard_020",
  "task_type": "sustained",
  "difficulty": "Hard",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a type of bird. List every bird you find.\n\nImportant: There may be similar-sounding items that are NOT a type of bird — do not include those.\n\n---\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe workshop on Kenji Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Among the items catalogued was a sparrow, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Dmitri fully understood. The wooden shelves bowed slightly under the weight.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A butterfly was noted in the margin of the inspector's report.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Among the items catalogued was a parrot, noted without further comment.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Celine recalled that a eagle had appeared briefly near the market.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The survey team documented a toucan in the area surrounding Zanzibar.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Dariush fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a dragonfly had been observed twice that week.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Amara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The logbook recorded a beetle at the northern edge of the district.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Reports from the harbour mentioned a pterodactyl had been observed twice that week.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Kaia mentioned seeing a heron while crossing the square.\n\nThe workshop on Adaeze Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Idris reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Willa reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA thin rain began to fall just as Nalini reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Trieste insisted on maintaining their position, while the merchants grew impatient.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The logbook recorded a osprey at the northern edge of the district.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Luang Prabang insisted on maintaining their position, while the merchants grew impatient. Among the items catalogued was a sugar glider, noted without further comment.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Reports from the harbour mentioned a finch had been observed twice that week.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The survey team documented a raven in the area surrounding Ulaanbaatar.\n\nThe workshop on Greta Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cusco insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Freya mentioned seeing a flying squirrel while crossing the square.\n\nA thin rain began to fall just as Xander reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Maren mentioned seeing a dove while crossing the square.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Zain reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Elio Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Idris fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Olena reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The survey team documented a puffin in the area surrounding Jaipur.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The logbook recorded a wasp at the northern edge of the district.\n\nThe workshop on Elio Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Elara recalled that a flying fish had appeared briefly near the market.\n---\n\nList ALL birds mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"sparrow\", \"parrot\", \"eagle\", \"toucan\", \"heron\", \"osprey\", \"finch\", \"raven\", \"dove\", \"puffin\"], \"nearmisses\": [\"butterfly\", \"dragonfly\", \"beetle\", \"pterodactyl\", \"sugar glider\", \"flying squirrel\", \"wasp\", \"flying fish\"]}"
 },
 {
  "task_id": "sustained_hard_021",
  "task_type": "sustained",
  "difficulty": "Hard",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river — do not include those.\n\n---\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Zora fully understood. The wooden shelves bowed slightly under the weight.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Among the items catalogued was a Bay of Bengal, noted without further comment.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Olena reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A Rhine was spotted near the old bridge that morning.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The logbook recorded a Zambezi at the northern edge of the district.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Colette reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Among the items catalogued was a Lake Victoria, noted without further comment.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Jaipur insisted on maintaining their position, while the merchants grew impatient.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Dmitri reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Among the items catalogued was a Congo, noted without further comment.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Joaquin fully understood. The wooden shelves bowed slightly under the weight. Among the items catalogued was a Nile, noted without further comment.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The logbook recorded a Dead Sea at the northern edge of the district.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The survey team documented a Lake Baikal in the area surrounding Fez.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The logbook recorded a Murray at the northern edge of the district.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Among the items catalogued was a Aral Sea, noted without further comment.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient. Among the items catalogued was a Strait of Gibraltar, noted without further comment.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Reports from the harbour mentioned a Oder had been observed twice that week.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A Tagus was noted in the margin of the inspector's report.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Hana fully understood. The wooden shelves bowed slightly under the weight. A Suez Canal was spotted near the old bridge that morning.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe workshop on Zora Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Reports from the harbour mentioned a Black Sea had been observed twice that week.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Joelle fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a Volga had been observed twice that week.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. A Ganges was noted in the margin of the inspector's report.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Among the items catalogued was a Mississippi, noted without further comment.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Rhine\", \"Zambezi\", \"Congo\", \"Nile\", \"Murray\", \"Oder\", \"Tagus\", \"Volga\", \"Ganges\", \"Mississippi\"], \"nearmisses\": [\"Bay of Bengal\", \"Lake Victoria\", \"Dead Sea\", \"Lake Baikal\", \"Aral Sea\", \"Strait of Gibraltar\", \"Suez Canal\", \"Black Sea\"]}"
 },
 {
  "task_id": "sustained_hard_022",
  "task_type": "sustained",
  "difficulty": "Hard",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a type of bird. List every bird you find.\n\nImportant: There may be similar-sounding items that are NOT a type of bird — do not include those.\n\n---\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A dove was spotted near the old bridge that morning.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Orla fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Kaia Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Among the items catalogued was a moth, noted without further comment.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Uma fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Runa recalled that a butterfly had appeared briefly near the market.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Reports from the harbour mentioned a toucan had been observed twice that week.\n\nThe old postal route between Oulu and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Freya recalled that a wasp had appeared briefly near the market.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Oulu insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Kaia fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Nalini fully understood. The wooden shelves bowed slightly under the weight. A parrot was noted in the margin of the inspector's report.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Uma recalled that a beetle had appeared briefly near the market.\n\nThe market square in Fez was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Among the items catalogued was a sugar glider, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Reports from the harbour mentioned a finch had been observed twice that week.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The logbook recorded a quail at the northern edge of the district.\n\nA thin rain began to fall just as Leif reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A oriole was spotted near the old bridge that morning.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Reports from the harbour mentioned a bat had been observed twice that week.\n\nA thin rain began to fall just as Leif reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Qadir Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The survey team documented a raven in the area surrounding Luang Prabang.\n\nThe workshop on Kenji Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. A eagle was noted in the margin of the inspector's report.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cusco insisted on maintaining their position, while the merchants grew impatient. Reports from the harbour mentioned a osprey had been observed twice that week.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Sigrid mentioned seeing a flying squirrel while crossing the square.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Paloma Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Maren mentioned seeing a falcon while crossing the square.\n\nA thin rain began to fall just as Freya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Uma recalled that a dragonfly had appeared briefly near the market.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n---\n\nList ALL birds mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"dove\", \"toucan\", \"parrot\", \"finch\", \"quail\", \"oriole\", \"raven\", \"eagle\", \"osprey\", \"falcon\"], \"nearmisses\": [\"moth\", \"butterfly\", \"wasp\", \"beetle\", \"sugar glider\", \"bat\", \"flying squirrel\", \"dragonfly\"]}"
 },
 {
  "task_id": "sustained_hard_023",
  "task_type": "sustained",
  "difficulty": "Hard",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a type of bird. List every bird you find.\n\nImportant: There may be similar-sounding items that are NOT a type of bird — do not include those.\n\n---\nA thin rain began to fall just as Zain reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Colette fully understood. The wooden shelves bowed slightly under the weight. A sugar glider was noted in the margin of the inspector's report.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Among the items catalogued was a eagle, noted without further comment.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Paloma mentioned seeing a raven while crossing the square.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Among the items catalogued was a magpie, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Nalini reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tariq fully understood. The wooden shelves bowed slightly under the weight.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Yara recalled that a parrot had appeared briefly near the market.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Kaia mentioned seeing a bat while crossing the square.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The logbook recorded a flying fish at the northern edge of the district.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Reports from the harbour mentioned a finch had been observed twice that week.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. A moth was spotted near the old bridge that morning.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe workshop on Greta Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Tala recalled that a wasp had appeared briefly near the market.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Yuki reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Elara fully understood. The wooden shelves bowed slightly under the weight. Zain mentioned seeing a ibis while crossing the square.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The survey team documented a woodpecker in the area surrounding Tbilisi.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Adaeze Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Haruto recalled that a dragonfly had appeared briefly near the market.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Cusco and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient. A pelican was noted in the margin of the inspector's report.\n\nThe workshop on Joaquin Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Reports from the harbour mentioned a dove had been observed twice that week.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cartagena insisted on maintaining their position, while the merchants grew impatient.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Among the items catalogued was a butterfly, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kotor insisted on maintaining their position, while the merchants grew impatient. The survey team documented a toucan in the area surrounding Zanzibar.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. A pterodactyl was noted in the margin of the inspector's report.\n---\n\nList ALL birds mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"eagle\", \"raven\", \"magpie\", \"parrot\", \"finch\", \"ibis\", \"woodpecker\", \"pelican\", \"dove\", \"toucan\"], \"nearmisses\": [\"sugar glider\", \"bat\", \"flying fish\", \"moth\", \"wasp\", \"dragonfly\", \"butterfly\", \"pterodactyl\"]}"
 },
 {
  "task_id": "sustained_expert_024",
  "task_type": "sustained",
  "difficulty": "Expert",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a type of bird. List every bird you find.\n\nImportant: There may be similar-sounding items that are NOT a type of bird — do not include those.\n\n---\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The logbook recorded a wasp at the northern edge of the district.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. A kingfisher was spotted near the old bridge that morning.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Viktor fully understood. The wooden shelves bowed slightly under the weight. A dragonfly was noted in the margin of the inspector's report.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Tariq recalled that a toucan had appeared briefly near the market.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tala fully understood. The wooden shelves bowed slightly under the weight.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Idris recalled that a beetle had appeared briefly near the market.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. A pterodactyl was spotted near the old bridge that morning.\n\nThe workshop on Idris Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A finch was spotted near the old bridge that morning.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The logbook recorded a butterfly at the northern edge of the district.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Kaia Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Orla fully understood. The wooden shelves bowed slightly under the weight.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Nalini reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Elio Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A puffin was spotted near the old bridge that morning.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. A oriole was spotted near the old bridge that morning.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tala fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Freya fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Colette fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Celine fully understood. The wooden shelves bowed slightly under the weight.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A flamingo was spotted near the old bridge that morning.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Nalini reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Dmitri fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A eagle was noted in the margin of the inspector's report.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Sigrid fully understood. The wooden shelves bowed slightly under the weight. Dmitri recalled that a sugar glider had appeared briefly near the market.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Zora fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient. Maren mentioned seeing a magpie while crossing the square.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The survey team documented a moth in the area surrounding Tallinn.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The logbook recorded a bat at the northern edge of the district.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Among the items catalogued was a starling, noted without further comment.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. A quail was spotted near the old bridge that morning.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Adaeze fully understood. The wooden shelves bowed slightly under the weight. The logbook recorded a ibis at the northern edge of the district.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A osprey was spotted near the old bridge that morning.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. A flying fish was spotted near the old bridge that morning.\n\nA thin rain began to fall just as Paloma reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Reports from the harbour mentioned a flying squirrel had been observed twice that week.\n---\n\nList ALL birds mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"kingfisher\", \"toucan\", \"finch\", \"puffin\", \"oriole\", \"flamingo\", \"eagle\", \"magpie\", \"starling\", \"quail\", \"ibis\", \"osprey\"], \"nearmisses\": [\"wasp\", \"dragonfly\", \"beetle\", \"pterodactyl\", \"butterfly\", \"sugar glider\", \"moth\", \"bat\", \"flying fish\", \"flying squirrel\"]}"
 },
 {
  "task_id": "sustained_expert_025",
  "task_type": "sustained",
  "difficulty": "Expert",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a type of bird. List every bird you find.\n\nImportant: There may be similar-sounding items that are NOT a type of bird — do not include those.\n\n---\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Kotor and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. A moth was spotted near the old bridge that morning.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Oulu and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Yuki reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Kaia fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a flamingo had been observed twice that week.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Paloma mentioned seeing a osprey while crossing the square.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Leif recalled that a sugar glider had appeared briefly near the market.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Reports from the harbour mentioned a raven had been observed twice that week.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A bat was spotted near the old bridge that morning.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Bashir fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Priya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Ugo mentioned seeing a beetle while crossing the square.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Colette mentioned seeing a butterfly while crossing the square.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA thin rain began to fall just as Bashir reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The logbook recorded a dragonfly at the northern edge of the district.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Maren mentioned seeing a wasp while crossing the square.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Luang Prabang insisted on maintaining their position, while the merchants grew impatient. Sigrid recalled that a puffin had appeared briefly near the market.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Amara fully understood. The wooden shelves bowed slightly under the weight. A quail was spotted near the old bridge that morning.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Reports from the harbour mentioned a oriole had been observed twice that week.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tariq fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Leif reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Joaquin Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The logbook recorded a magpie at the northern edge of the district.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Adaeze mentioned seeing a pterodactyl while crossing the square.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. A sparrow was noted in the margin of the inspector's report.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Vesna reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Joelle fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a woodpecker had been observed twice that week.\n\nThe market square in Tallinn was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Reports from the harbour mentioned a flying squirrel had been observed twice that week.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Nalini fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Soren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Reports from the harbour mentioned a kingfisher had been observed twice that week.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Reports from the harbour mentioned a flying fish had been observed twice that week.\n\nA thin rain began to fall just as Tala reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. A dove was spotted near the old bridge that morning.\n\nThe workshop on Wren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The logbook recorded a falcon at the northern edge of the district.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n---\n\nList ALL birds mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"flamingo\", \"osprey\", \"raven\", \"puffin\", \"quail\", \"oriole\", \"magpie\", \"sparrow\", \"woodpecker\", \"kingfisher\", \"dove\", \"falcon\"], \"nearmisses\": [\"moth\", \"sugar glider\", \"bat\", \"beetle\", \"butterfly\", \"dragonfly\", \"wasp\", \"pterodactyl\", \"flying squirrel\", \"flying fish\"]}"
 },
 {
  "task_id": "sustained_expert_026",
  "task_type": "sustained",
  "difficulty": "Expert",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river — do not include those.\n\n---\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The survey team documented a Panama Canal in the area surrounding Fez.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Qadir recalled that a Yangtze had appeared briefly near the market.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Among the items catalogued was a Elbe, noted without further comment.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient. Yara recalled that a Ganges had appeared briefly near the market.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The survey team documented a Lake Victoria in the area surrounding Kotor.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A Amazon was noted in the margin of the inspector's report.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Magnus recalled that a Lake Baikal had appeared briefly near the market.\n\nA thin rain began to fall just as Greta reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tariq fully understood. The wooden shelves bowed slightly under the weight.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The logbook recorded a Strait of Gibraltar at the northern edge of the district.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yuki fully understood. The wooden shelves bowed slightly under the weight. Sigrid recalled that a Volga had appeared briefly near the market.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Lumi recalled that a Don had appeared briefly near the market.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Among the items catalogued was a Murray, noted without further comment.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient. Elio mentioned seeing a Black Sea while crossing the square.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Runa recalled that a Caspian Sea had appeared briefly near the market.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. A Zambezi was noted in the margin of the inspector's report.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Zora fully understood. The wooden shelves bowed slightly under the weight.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Tala reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The survey team documented a Euphrates in the area surrounding Cartagena.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Kenji reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A Rhine was noted in the margin of the inspector's report.\n\nA thin rain began to fall just as Haruto reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Vesna fully understood. The wooden shelves bowed slightly under the weight. Magnus mentioned seeing a Nile while crossing the square.\n\nThe old postal route between Cusco and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A Dead Sea was noted in the margin of the inspector's report.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The survey team documented a Aral Sea in the area surrounding Reykjavik.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. A Suez Canal was spotted near the old bridge that morning.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tariq fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Greta Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kumasi insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Jaipur insisted on maintaining their position, while the merchants grew impatient. Among the items catalogued was a Tagus, noted without further comment.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yuki fully understood. The wooden shelves bowed slightly under the weight.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Freya recalled that a Bay of Bengal had appeared briefly near the market.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Yangtze\", \"Elbe\", \"Ganges\", \"Amazon\", \"Volga\", \"Don\", \"Murray\", \"Zambezi\", \"Euphrates\", \"Rhine\", \"Nile\", \"Tagus\"], \"nearmisses\": [\"Panama Canal\", \"Lake Victoria\", \"Lake Baikal\", \"Strait of Gibraltar\", \"Black Sea\", \"Caspian Sea\", \"Dead Sea\", \"Aral Sea\", \"Suez Canal\", \"Bay of Bengal\"]}"
 },
 {
  "task_id": "sustained_expert_027",
  "task_type": "sustained",
  "difficulty": "Expert",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument — do not include those.\n\n---\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. A tabla was spotted near the old bridge that morning.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Willa reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Joaquin Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tariq fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Tala recalled that a speaker had appeared briefly near the market.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cartagena insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Ugo recalled that a zither had appeared briefly near the market.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Reports from the harbour mentioned a microphone had been observed twice that week.\n\nThe workshop on Olena Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Idris Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Nico Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The logbook recorded a pitch pipe at the northern edge of the district.\n\nA thin rain began to fall just as Ugo reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Elara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Willa mentioned seeing a music stand while crossing the square.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Leif reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. A bassoon was noted in the margin of the inspector's report.\n\nA thin rain began to fall just as Elara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Vesna Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. A metronome was noted in the margin of the inspector's report.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The survey team documented a tuning fork in the area surrounding Jaipur.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Among the items catalogued was a mixer, noted without further comment.\n\nA thin rain began to fall just as Zora reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Ines Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Among the items catalogued was a record player, noted without further comment.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Celine fully understood. The wooden shelves bowed slightly under the weight. Celine mentioned seeing a erhu while crossing the square.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. A sitar was noted in the margin of the inspector's report.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Zanzibar insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Reports from the harbour mentioned a timpani had been observed twice that week.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Hana recalled that a headphones had appeared briefly near the market.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The logbook recorded a violin at the northern edge of the district.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. A oboe was noted in the margin of the inspector's report.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A amplifier was spotted near the old bridge that morning.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe old postal route between Recife and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Reports from the harbour mentioned a balalaika had been observed twice that week.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Luang Prabang insisted on maintaining their position, while the merchants grew impatient.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Ugo recalled that a dulcimer had appeared briefly near the market.\n\nThe workshop on Runa Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Among the items catalogued was a oud, noted without further comment.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Colette mentioned seeing a theremin while crossing the square.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"tabla\", \"zither\", \"bassoon\", \"erhu\", \"sitar\", \"timpani\", \"violin\", \"oboe\", \"balalaika\", \"dulcimer\", \"oud\", \"theremin\"], \"nearmisses\": [\"speaker\", \"microphone\", \"pitch pipe\", \"music stand\", \"metronome\", \"tuning fork\", \"mixer\", \"record player\", \"headphones\", \"amplifier\"]}"
 },
 {
  "task_id": "sustained_expert_028",
  "task_type": "sustained",
  "difficulty": "Expert",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river — do not include those.\n\n---\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Haruto mentioned seeing a Aral Sea while crossing the square.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Nalini fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Magnus Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cusco insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Leif reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Xander reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Reports from the harbour mentioned a Lake Victoria had been observed twice that week.\n\nA thin rain began to fall just as Uma reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. A Oder was spotted near the old bridge that morning.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient. A Lake Baikal was noted in the margin of the inspector's report.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The survey team documented a Mekong in the area surrounding Tbilisi.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Ravi fully understood. The wooden shelves bowed slightly under the weight.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Freya fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Tariq reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Elara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Nalini Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The logbook recorded a Black Sea at the northern edge of the district.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Nico recalled that a Strait of Gibraltar had appeared briefly near the market.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Idris fully understood. The wooden shelves bowed slightly under the weight. A Indus was spotted near the old bridge that morning.\n\nA thin rain began to fall just as Yara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The survey team documented a Bay of Bengal in the area surrounding Bruges.\n\nThe workshop on Uma Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Reports from the harbour mentioned a Murray had been observed twice that week.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A Congo was noted in the margin of the inspector's report.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA thin rain began to fall just as Soren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Kaia mentioned seeing a Volga while crossing the square.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Among the items catalogued was a Panama Canal, noted without further comment.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Among the items catalogued was a Suez Canal, noted without further comment.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Xander recalled that a Don had appeared briefly near the market.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Reports from the harbour mentioned a Dead Sea had been observed twice that week.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Zanzibar insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA thin rain began to fall just as Dmitri reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Olena reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Valetta was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The logbook recorded a Caspian Sea at the northern edge of the district.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Vesna reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Reports from the harbour mentioned a Euphrates had been observed twice that week.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Tala Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The survey team documented a Amazon in the area surrounding Bruges.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Reports from the harbour mentioned a Mississippi had been observed twice that week.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient. A Zambezi was noted in the margin of the inspector's report.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Among the items catalogued was a Tigris, noted without further comment.\n\nThe market square in Valetta was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA thin rain began to fall just as Lumi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Oder\", \"Mekong\", \"Indus\", \"Murray\", \"Congo\", \"Volga\", \"Don\", \"Euphrates\", \"Amazon\", \"Mississippi\", \"Zambezi\", \"Tigris\"], \"nearmisses\": [\"Aral Sea\", \"Lake Victoria\", \"Lake Baikal\", \"Black Sea\", \"Strait of Gibraltar\", \"Bay of Bengal\", \"Panama Canal\", \"Suez Canal\", \"Dead Sea\", \"Caspian Sea\"]}"
 },
 {
  "task_id": "sustained_expert_029",
  "task_type": "sustained",
  "difficulty": "Expert",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument — do not include those.\n\n---\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Dariush Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. A mandolin was noted in the margin of the inspector's report.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Celine fully understood. The wooden shelves bowed slightly under the weight. The survey team documented a koto in the area surrounding Fez.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yuki fully understood. The wooden shelves bowed slightly under the weight.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Uma recalled that a record player had appeared briefly near the market.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe workshop on Orla Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Priya recalled that a theremin had appeared briefly near the market.\n\nThe workshop on Amara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Celine reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The survey team documented a pitch pipe in the area surrounding Oulu.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Lumi fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Ines recalled that a harp had appeared briefly near the market.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA thin rain began to fall just as Soren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The survey team documented a headphones in the area surrounding Plovdiv.\n\nA thin rain began to fall just as Dariush reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Tariq mentioned seeing a amplifier while crossing the square.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Colette fully understood. The wooden shelves bowed slightly under the weight. A mixer was noted in the margin of the inspector's report.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Joelle fully understood. The wooden shelves bowed slightly under the weight.\n\nThe workshop on Zain Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Tariq recalled that a tuning fork had appeared briefly near the market.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient. Magnus mentioned seeing a speaker while crossing the square.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Joelle fully understood. The wooden shelves bowed slightly under the weight.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Dariush recalled that a dulcimer had appeared briefly near the market.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Lumi fully understood. The wooden shelves bowed slightly under the weight.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Hana recalled that a flute had appeared briefly near the market.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Cusco and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tariq fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Bashir fully understood. The wooden shelves bowed slightly under the weight. A zither was spotted near the old bridge that morning.\n\nA thin rain began to fall just as Ravi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The survey team documented a tabla in the area surrounding Cartagena.\n\nThe old postal route between Jaipur and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A microphone was noted in the margin of the inspector's report.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. The survey team documented a violin in the area surrounding Oulu.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Willa fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Reports from the harbour mentioned a metronome had been observed twice that week.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Dmitri fully understood. The wooden shelves bowed slightly under the weight.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. A timpani was noted in the margin of the inspector's report.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The survey team documented a cello in the area surrounding Cartagena.\n\nA thin rain began to fall just as Magnus reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Tala Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Yuki recalled that a hurdy-gurdy had appeared briefly near the market.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe workshop on Nalini Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient. A music stand was noted in the margin of the inspector's report.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Trieste insisted on maintaining their position, while the merchants grew impatient.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"mandolin\", \"koto\", \"theremin\", \"harp\", \"dulcimer\", \"flute\", \"zither\", \"tabla\", \"violin\", \"timpani\", \"cello\", \"hurdy-gurdy\"], \"nearmisses\": [\"record player\", \"pitch pipe\", \"headphones\", \"amplifier\", \"mixer\", \"tuning fork\", \"speaker\", \"microphone\", \"metronome\", \"music stand\"]}"
 },
 {
  "task_id": "sustained_expert_030",
  "task_type": "sustained",
  "difficulty": "Expert",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument — do not include those.\n\n---\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Kenji Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Femi recalled that a theremin had appeared briefly near the market.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Oulu insisted on maintaining their position, while the merchants grew impatient. The logbook recorded a balalaika at the northern edge of the district.\n\nA thin rain began to fall just as Sigrid reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Valetta was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A zither was spotted near the old bridge that morning.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Paloma fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a sitar had been observed twice that week.\n\nThe workshop on Nico Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Among the items catalogued was a erhu, noted without further comment.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A pitch pipe was spotted near the old bridge that morning.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Qadir mentioned seeing a amplifier while crossing the square.\n\nThe workshop on Dariush Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The logbook recorded a record player at the northern edge of the district.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Nico reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Fez was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Haruto recalled that a timpani had appeared briefly near the market.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Reports from the harbour mentioned a speaker had been observed twice that week.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The logbook recorded a metronome at the northern edge of the district.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient. Joaquin mentioned seeing a bassoon while crossing the square.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. A mbira was spotted near the old bridge that morning.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Olena reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Sigrid reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cartagena insisted on maintaining their position, while the merchants grew impatient. The logbook recorded a microphone at the northern edge of the district.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. A dulcimer was spotted near the old bridge that morning.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. A mandolin was spotted near the old bridge that morning.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Reports from the harbour mentioned a tabla had been observed twice that week.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A music stand was spotted near the old bridge that morning.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Priya Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The logbook recorded a mixer at the northern edge of the district.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A tuning fork was spotted near the old bridge that morning.\n\nThe workshop on Greta Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. A flute was spotted near the old bridge that morning.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Luang Prabang insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe market square in Tallinn was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. A headphones was noted in the margin of the inspector's report.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Celine fully understood. The wooden shelves bowed slightly under the weight.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"theremin\", \"balalaika\", \"zither\", \"sitar\", \"erhu\", \"timpani\", \"bassoon\", \"mbira\", \"dulcimer\", \"mandolin\", \"tabla\", \"flute\"], \"nearmisses\": [\"pitch pipe\", \"amplifier\", \"record player\", \"speaker\", \"metronome\", \"microphone\", \"music stand\", \"mixer\", \"tuning fork\", \"headphones\"]}"
 },
 {
  "task_id": "sustained_expert_031",
  "task_type": "sustained",
  "difficulty": "Expert",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a metallic element. List every metal you find.\n\nImportant: There may be similar-sounding items that are NOT a metallic element — do not include those.\n\n---\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Sigrid reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Nico recalled that a rhodium had appeared briefly near the market.\n\nThe market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Wren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Among the items catalogued was a ceramic, noted without further comment.\n\nThe market square in Fez was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Yara mentioned seeing a platinum while crossing the square.\n\nThe workshop on Runa Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Idris recalled that a granite had appeared briefly near the market.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The survey team documented a chalk in the area surrounding Fez.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Paloma fully understood. The wooden shelves bowed slightly under the weight.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Fez was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Among the items catalogued was a palladium, noted without further comment.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Greta reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The survey team documented a manganese in the area surrounding Mandalay.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Xander fully understood. The wooden shelves bowed slightly under the weight. The survey team documented a wood in the area surrounding Mandalay.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. A sand was spotted near the old bridge that morning.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Paloma reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Dmitri Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The logbook recorded a tin at the northern edge of the district.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tariq fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Ugo reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. A chromium was noted in the margin of the inspector's report.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Elara fully understood. The wooden shelves bowed slightly under the weight.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kumasi insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Elio fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Reports from the harbour mentioned a plastic had been observed twice that week.\n\nA thin rain began to fall just as Orla reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Paloma recalled that a iridium had appeared briefly near the market.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Kenji fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Sigrid fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Idris reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yara fully understood. The wooden shelves bowed slightly under the weight.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe workshop on Gael Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Colette recalled that a vanadium had appeared briefly near the market.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Reports from the harbour mentioned a rubber had been observed twice that week.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The logbook recorded a glass at the northern edge of the district.\n\nThe workshop on Haruto Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The survey team documented a cobalt in the area surrounding Gdansk.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The survey team documented a osmium in the area surrounding Plovdiv.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Lumi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The survey team documented a ruthenium in the area surrounding Cartagena.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Vesna Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Reports from the harbour mentioned a concrete had been observed twice that week.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Jaipur and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. A marble was spotted near the old bridge that morning.\n\nThe workshop on Femi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Among the items catalogued was a lead, noted without further comment.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n---\n\nList ALL metals mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"rhodium\", \"platinum\", \"palladium\", \"manganese\", \"tin\", \"chromium\", \"iridium\", \"vanadium\", \"cobalt\", \"osmium\", \"ruthenium\", \"lead\"], \"nearmisses\": [\"ceramic\", \"granite\", \"chalk\", \"wood\", \"sand\", \"plastic\", \"rubber\", \"glass\", \"concrete\", \"marble\"]}"
 },
 {
  "task_id": "sustained_frontier_032",
  "task_type": "sustained",
  "difficulty": "Frontier",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river — do not include those.\n\n---\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Idris fully understood. The wooden shelves bowed slightly under the weight.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Priya Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A Don was spotted near the old bridge that morning.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A Dead Sea was noted in the margin of the inspector's report.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Colette fully understood. The wooden shelves bowed slightly under the weight. Among the items catalogued was a Lake Baikal, noted without further comment.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Bashir reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient. Zain recalled that a Euphrates had appeared briefly near the market.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tala fully understood. The wooden shelves bowed slightly under the weight. The logbook recorded a Mekong at the northern edge of the district.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kumasi insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Cusco and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Yara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe workshop on Joaquin Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe workshop on Ines Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Orla recalled that a Black Sea had appeared briefly near the market.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Nalini reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A Bay of Bengal was spotted near the old bridge that morning.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Leif fully understood. The wooden shelves bowed slightly under the weight.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A Loire was noted in the margin of the inspector's report.\n\nThe market square in Tallinn was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Jaipur and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Femi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Viktor reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kotor insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Among the items catalogued was a Panama Canal, noted without further comment.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. A Lake Victoria was noted in the margin of the inspector's report.\n\nA thin rain began to fall just as Ugo reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Among the items catalogued was a Amazon, noted without further comment.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A Aral Sea was noted in the margin of the inspector's report.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe workshop on Viktor Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Idris reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe workshop on Magnus Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The logbook recorded a Suez Canal at the northern edge of the district.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Leif Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A Tagus was spotted near the old bridge that morning.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Adaeze reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Luang Prabang insisted on maintaining their position, while the merchants grew impatient.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Freya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Haruto reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Femi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Reports from the harbour mentioned a Mississippi had been observed twice that week.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe workshop on Femi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Dariush mentioned seeing a Strait of Gibraltar while crossing the square.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Soren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe workshop on Dariush Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Nico reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Zain fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The survey team documented a Caspian Sea in the area surrounding Kumasi.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Hana Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. A Murray was spotted near the old bridge that morning.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Idris fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a Rhine had been observed twice that week.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Dariush Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Greta reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Wren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. The survey team documented a Zambezi in the area surrounding Recife.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Don\", \"Euphrates\", \"Mekong\", \"Loire\", \"Amazon\", \"Tagus\", \"Mississippi\", \"Murray\", \"Rhine\", \"Zambezi\"], \"nearmisses\": [\"Dead Sea\", \"Lake Baikal\", \"Black Sea\", \"Bay of Bengal\", \"Panama Canal\", \"Lake Victoria\", \"Aral Sea\", \"Suez Canal\", \"Strait of Gibraltar\", \"Caspian Sea\"]}"
 },
 {
  "task_id": "sustained_frontier_033",
  "task_type": "sustained",
  "difficulty": "Frontier",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a metallic element. List every metal you find.\n\nImportant: There may be similar-sounding items that are NOT a metallic element — do not include those.\n\n---\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Haruto Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tariq fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Viktor fully understood. The wooden shelves bowed slightly under the weight.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Sigrid recalled that a cobalt had appeared briefly near the market.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Soren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe workshop on Ines Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Uma fully understood. The wooden shelves bowed slightly under the weight.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Jaipur and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Trieste insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Ines Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Among the items catalogued was a niobium, noted without further comment.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Nalini Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. A ruthenium was noted in the margin of the inspector's report.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. A glass was spotted near the old bridge that morning.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Femi fully understood. The wooden shelves bowed slightly under the weight.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Yara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Greta Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Magnus reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA thin rain began to fall just as Gael reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Tariq recalled that a manganese had appeared briefly near the market.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA thin rain began to fall just as Idris reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Soren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A vanadium was noted in the margin of the inspector's report.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Haruto fully understood. The wooden shelves bowed slightly under the weight. Yara recalled that a tin had appeared briefly near the market.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Lumi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Greta reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Vesna Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Kenji mentioned seeing a concrete while crossing the square.\n\nA thin rain began to fall just as Bashir reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The logbook recorded a rubber at the northern edge of the district.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. A lead was noted in the margin of the inspector's report.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Amara fully understood. The wooden shelves bowed slightly under the weight.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Priya mentioned seeing a chromium while crossing the square.\n\nThe market square in Tallinn was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Cusco and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Uma fully understood. The wooden shelves bowed slightly under the weight.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The survey team documented a ceramic in the area surrounding Tallinn.\n\nA thin rain began to fall just as Zain reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Ugo Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Joelle reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. A sand was noted in the margin of the inspector's report.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Greta recalled that a chalk had appeared briefly near the market.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Orla reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Lumi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Gael recalled that a granite had appeared briefly near the market.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Colette mentioned seeing a rhodium while crossing the square.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Reports from the harbour mentioned a marble had been observed twice that week.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Elara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Vesna Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The logbook recorded a wood at the northern edge of the district.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Among the items catalogued was a plastic, noted without further comment.\n\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA thin rain began to fall just as Freya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Reports from the harbour mentioned a iridium had been observed twice that week.\n---\n\nList ALL metals mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"cobalt\", \"niobium\", \"ruthenium\", \"manganese\", \"vanadium\", \"tin\", \"lead\", \"chromium\", \"rhodium\", \"iridium\"], \"nearmisses\": [\"glass\", \"concrete\", \"rubber\", \"ceramic\", \"sand\", \"chalk\", \"granite\", \"marble\", \"wood\", \"plastic\"]}"
 },
 {
  "task_id": "sustained_frontier_034",
  "task_type": "sustained",
  "difficulty": "Frontier",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument — do not include those.\n\n---\nA thin rain began to fall just as Bashir reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Among the items catalogued was a music stand, noted without further comment.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Olena reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Amara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Reports from the harbour mentioned a amplifier had been observed twice that week.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. A koto was spotted near the old bridge that morning.\n\nA thin rain began to fall just as Ugo reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Oulu and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Dmitri reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Colette fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Xander fully understood. The wooden shelves bowed slightly under the weight.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. A microphone was spotted near the old bridge that morning.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Wren mentioned seeing a bassoon while crossing the square.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Among the items catalogued was a headphones, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient. A speaker was noted in the margin of the inspector's report.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Uma fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yara fully understood. The wooden shelves bowed slightly under the weight.\n\nThe workshop on Wren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Recife and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Adaeze reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Tallinn was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Among the items catalogued was a metronome, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The logbook recorded a timpani at the northern edge of the district.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Dariush reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Reports from the harbour mentioned a dulcimer had been observed twice that week.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cusco insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Nalini reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Joelle fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Trieste insisted on maintaining their position, while the merchants grew impatient.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Among the items catalogued was a lute, noted without further comment.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Gael reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Among the items catalogued was a balalaika, noted without further comment.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Reports from the harbour mentioned a tuning fork had been observed twice that week.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Amara fully understood. The wooden shelves bowed slightly under the weight.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A mixer was noted in the margin of the inspector's report.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe workshop on Nico Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Among the items catalogued was a erhu, noted without further comment.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Reports from the harbour mentioned a oud had been observed twice that week.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Idris fully understood. The wooden shelves bowed slightly under the weight.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Freya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Qadir Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The survey team documented a record player in the area surrounding Gdansk.\n\nThe old postal route between Oulu and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Kaia reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Willa Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The logbook recorded a mandolin at the northern edge of the district.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Kaia Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Celine fully understood. The wooden shelves bowed slightly under the weight.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Trieste insisted on maintaining their position, while the merchants grew impatient.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Dmitri reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Idris Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Maren fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Ugo reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Reports from the harbour mentioned a zither had been observed twice that week.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Haruto recalled that a pitch pipe had appeared briefly near the market.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Amara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe workshop on Wren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"koto\", \"bassoon\", \"timpani\", \"dulcimer\", \"lute\", \"balalaika\", \"erhu\", \"oud\", \"mandolin\", \"zither\"], \"nearmisses\": [\"music stand\", \"amplifier\", \"microphone\", \"headphones\", \"speaker\", \"metronome\", \"tuning fork\", \"mixer\", \"record player\", \"pitch pipe\"]}"
 },
 {
  "task_id": "sustained_frontier_035",
  "task_type": "sustained",
  "difficulty": "Frontier",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument — do not include those.\n\n---\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A amplifier was spotted near the old bridge that morning.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Xander fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Zora reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Reports from the harbour mentioned a zither had been observed twice that week.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA thin rain began to fall just as Ugo reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A timpani was noted in the margin of the inspector's report.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA thin rain began to fall just as Vesna reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kotor insisted on maintaining their position, while the merchants grew impatient. A mbira was noted in the margin of the inspector's report.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Ravi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kotor insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Ugo Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Joelle recalled that a tabla had appeared briefly near the market.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Lumi recalled that a pitch pipe had appeared briefly near the market.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Joaquin reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Recife and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Joelle reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Elara fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Wren fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a speaker had been observed twice that week.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Maren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe workshop on Yara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Gael recalled that a hurdy-gurdy had appeared briefly near the market.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Celine fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Femi fully understood. The wooden shelves bowed slightly under the weight. The survey team documented a balalaika in the area surrounding Trieste.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Joelle reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kotor insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Freya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Zanzibar insisted on maintaining their position, while the merchants grew impatient.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The logbook recorded a mixer at the northern edge of the district.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Nico reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Zora reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The survey team documented a microphone in the area surrounding Valetta.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Yara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Gael Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. A metronome was noted in the margin of the inspector's report.\n\nThe workshop on Kaia Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Hana mentioned seeing a oud while crossing the square.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Nalini recalled that a theremin had appeared briefly near the market.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Among the items catalogued was a sitar, noted without further comment.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Soren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. A headphones was spotted near the old bridge that morning.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The survey team documented a record player in the area surrounding Kumasi.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Joelle reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Maren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Freya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Magnus fully understood. The wooden shelves bowed slightly under the weight.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA thin rain began to fall just as Femi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Hana fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Celine fully understood. The wooden shelves bowed slightly under the weight.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Runa Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Jaipur insisted on maintaining their position, while the merchants grew impatient.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient. Sigrid mentioned seeing a tuning fork while crossing the square.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The logbook recorded a bassoon at the northern edge of the district.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Colette fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Oulu insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Orla reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The logbook recorded a music stand at the northern edge of the district.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"zither\", \"timpani\", \"mbira\", \"tabla\", \"hurdy-gurdy\", \"balalaika\", \"oud\", \"theremin\", \"sitar\", \"bassoon\"], \"nearmisses\": [\"amplifier\", \"pitch pipe\", \"speaker\", \"mixer\", \"microphone\", \"metronome\", \"headphones\", \"record player\", \"tuning fork\", \"music stand\"]}"
 },
 {
  "task_id": "sustained_frontier_036",
  "task_type": "sustained",
  "difficulty": "Frontier",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument — do not include those.\n\n---\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Among the items catalogued was a harp, noted without further comment.\n\nA thin rain began to fall just as Runa reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Kenji Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The survey team documented a tuning fork in the area surrounding Plovdiv.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Among the items catalogued was a erhu, noted without further comment.\n\nA thin rain began to fall just as Ravi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Ravi fully understood. The wooden shelves bowed slightly under the weight.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Reports from the harbour mentioned a metronome had been observed twice that week.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Gael fully understood. The wooden shelves bowed slightly under the weight. Kaia recalled that a zither had appeared briefly near the market.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Soren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. A music stand was spotted near the old bridge that morning.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. A lute was spotted near the old bridge that morning.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Kaia fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Priya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Dmitri Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Qadir reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Adaeze fully understood. The wooden shelves bowed slightly under the weight. Yara mentioned seeing a microphone while crossing the square.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Viktor Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A mixer was spotted near the old bridge that morning.\n\nThe workshop on Dmitri Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The survey team documented a balalaika in the area surrounding Ulaanbaatar.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Sigrid fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Leif fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a mandolin had been observed twice that week.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cusco insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Among the items catalogued was a flute, noted without further comment.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yuki fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Amara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Among the items catalogued was a timpani, noted without further comment.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The survey team documented a violin in the area surrounding Reykjavik.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. A dulcimer was spotted near the old bridge that morning.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Wren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Among the items catalogued was a record player, noted without further comment.\n\nA thin rain began to fall just as Nico reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Runa recalled that a pitch pipe had appeared briefly near the market.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Zain Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Kotor and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Zain reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Yuki reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Bram Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Reports from the harbour mentioned a speaker had been observed twice that week.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA thin rain began to fall just as Kaia reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Elara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Recife and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Runa reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Amara fully understood. The wooden shelves bowed slightly under the weight.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Xander Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. A amplifier was spotted near the old bridge that morning.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Hana fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Lumi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Bram reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Kenji reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Uma fully understood. The wooden shelves bowed slightly under the weight. Among the items catalogued was a headphones, noted without further comment.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"harp\", \"erhu\", \"zither\", \"lute\", \"balalaika\", \"mandolin\", \"flute\", \"timpani\", \"violin\", \"dulcimer\"], \"nearmisses\": [\"tuning fork\", \"metronome\", \"music stand\", \"microphone\", \"mixer\", \"record player\", \"pitch pipe\", \"speaker\", \"amplifier\", \"headphones\"]}"
 },
 {
  "task_id": "sustained_frontier_037",
  "task_type": "sustained",
  "difficulty": "Frontier",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river — do not include those.\n\n---\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Qadir fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA thin rain began to fall just as Femi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. A Danube was spotted near the old bridge that morning.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Adaeze fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Cusco and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient. A Lake Victoria was spotted near the old bridge that morning.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Zanzibar insisted on maintaining their position, while the merchants grew impatient.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Nalini recalled that a Tagus had appeared briefly near the market.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Dmitri reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Willa reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Among the items catalogued was a Euphrates, noted without further comment.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The survey team documented a Caspian Sea in the area surrounding Cartagena.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Amara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Hana fully understood. The wooden shelves bowed slightly under the weight.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Sigrid recalled that a Ganges had appeared briefly near the market.\n\nThe old postal route between Jaipur and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Haruto Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe old postal route between Kotor and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The survey team documented a Elbe in the area surrounding Luang Prabang.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Paloma fully understood. The wooden shelves bowed slightly under the weight.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Among the items catalogued was a Strait of Gibraltar, noted without further comment.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Dmitri fully understood. The wooden shelves bowed slightly under the weight.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Vesna Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Zain Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe market square in Cusco was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA thin rain began to fall just as Ravi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Joaquin mentioned seeing a Black Sea while crossing the square.\n\nThe workshop on Qadir Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe workshop on Magnus Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe market square in Cusco was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Among the items catalogued was a Suez Canal, noted without further comment.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Priya recalled that a Don had appeared briefly near the market.\n\nThe workshop on Lumi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Orla reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Willa reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Hana fully understood. The wooden shelves bowed slightly under the weight. Among the items catalogued was a Lake Baikal, noted without further comment.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Reports from the harbour mentioned a Bay of Bengal had been observed twice that week.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Kenji Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Celine Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Colette recalled that a Indus had appeared briefly near the market.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Vesna mentioned seeing a Tigris while crossing the square.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Ugo Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Kenji fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yuki fully understood. The wooden shelves bowed slightly under the weight.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Amara fully understood. The wooden shelves bowed slightly under the weight.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The logbook recorded a Panama Canal at the northern edge of the district.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient. A Yangtze was spotted near the old bridge that morning.\n\nA thin rain began to fall just as Celine reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Idris reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kotor insisted on maintaining their position, while the merchants grew impatient.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Runa fully understood. The wooden shelves bowed slightly under the weight. The survey team documented a Aral Sea in the area surrounding Mandalay.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Femi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Zora fully understood. The wooden shelves bowed slightly under the weight.\n\nThe workshop on Yara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Tariq recalled that a Mississippi had appeared briefly near the market.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Among the items catalogued was a Dead Sea, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Danube\", \"Tagus\", \"Euphrates\", \"Ganges\", \"Elbe\", \"Don\", \"Indus\", \"Tigris\", \"Yangtze\", \"Mississippi\"], \"nearmisses\": [\"Lake Victoria\", \"Caspian Sea\", \"Strait of Gibraltar\", \"Black Sea\", \"Suez Canal\", \"Lake Baikal\", \"Bay of Bengal\", \"Panama Canal\", \"Aral Sea\", \"Dead Sea\"]}"
 },
 {
  "task_id": "sustained_frontier_038",
  "task_type": "sustained",
  "difficulty": "Frontier",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument — do not include those.\n\n---\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Jaipur insisted on maintaining their position, while the merchants grew impatient.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A speaker was spotted near the old bridge that morning.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Reports from the harbour mentioned a record player had been observed twice that week.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Reports from the harbour mentioned a tuning fork had been observed twice that week.\n\nThe workshop on Nico Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe workshop on Yara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cartagena insisted on maintaining their position, while the merchants grew impatient.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Among the items catalogued was a music stand, noted without further comment.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Reports from the harbour mentioned a headphones had been observed twice that week.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Viktor fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Dariush recalled that a tabla had appeared briefly near the market.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe workshop on Dmitri Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Elio recalled that a zither had appeared briefly near the market.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Yara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Among the items catalogued was a mandolin, noted without further comment.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Zora reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe old postal route between Oulu and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Maren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Haruto reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Jaipur insisted on maintaining their position, while the merchants grew impatient.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Elio reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA thin rain began to fall just as Tariq reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Sigrid fully understood. The wooden shelves bowed slightly under the weight.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe workshop on Runa Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. A oud was spotted near the old bridge that morning.\n\nThe workshop on Colette Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Gael mentioned seeing a lute while crossing the square.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Priya fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The survey team documented a pitch pipe in the area surrounding Kotor.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA thin rain began to fall just as Magnus reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Colette reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Kenji reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Dmitri recalled that a harp had appeared briefly near the market.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Elio fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The logbook recorded a amplifier at the northern edge of the district.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Reports from the harbour mentioned a mixer had been observed twice that week.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Reports from the harbour mentioned a metronome had been observed twice that week.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Leif fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient. Lumi mentioned seeing a mbira while crossing the square.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Ines reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Ugo mentioned seeing a erhu while crossing the square.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Paloma mentioned seeing a sitar while crossing the square.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cusco insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kotor insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Kaia Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The survey team documented a flute in the area surrounding Jaipur.\n\nThe workshop on Olena Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Hana fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Soren fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Greta reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Yara mentioned seeing a microphone while crossing the square.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"tabla\", \"zither\", \"mandolin\", \"oud\", \"lute\", \"harp\", \"mbira\", \"erhu\", \"sitar\", \"flute\"], \"nearmisses\": [\"speaker\", \"record player\", \"tuning fork\", \"music stand\", \"headphones\", \"pitch pipe\", \"amplifier\", \"mixer\", \"metronome\", \"microphone\"]}"
 },
 {
  "task_id": "sustained_frontier_039",
  "task_type": "sustained",
  "difficulty": "Frontier",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a type of bird. List every bird you find.\n\nImportant: There may be similar-sounding items that are NOT a type of bird — do not include those.\n\n---\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Lumi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Nico Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. A dove was noted in the margin of the inspector's report.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The survey team documented a wasp in the area surrounding Ulaanbaatar.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Gael reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Orla fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Orla reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Zora Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Colette fully understood. The wooden shelves bowed slightly under the weight.\n\nThe workshop on Soren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe workshop on Celine Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Soren fully understood. The wooden shelves bowed slightly under the weight.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Fez was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Among the items catalogued was a osprey, noted without further comment.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Bram Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A flying squirrel was noted in the margin of the inspector's report.\n\nThe workshop on Tala Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The survey team documented a flying fish in the area surrounding Plovdiv.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Uma Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Reports from the harbour mentioned a dragonfly had been observed twice that week.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Zora reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Paloma Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Colette Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Olena reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA thin rain began to fall just as Femi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Ines fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Joaquin fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Jaipur insisted on maintaining their position, while the merchants grew impatient.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The logbook recorded a quail at the northern edge of the district.\n\nThe workshop on Tala Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Greta fully understood. The wooden shelves bowed slightly under the weight. Among the items catalogued was a magpie, noted without further comment.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A raven was noted in the margin of the inspector's report.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient. A toucan was noted in the margin of the inspector's report.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Hana Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cusco insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Priya Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A pelican was spotted near the old bridge that morning.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Among the items catalogued was a pterodactyl, noted without further comment.\n\nThe market square in Valetta was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Priya fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Priya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Uma recalled that a parrot had appeared briefly near the market.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Among the items catalogued was a bat, noted without further comment.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Kenji reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Fez was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A beetle was noted in the margin of the inspector's report.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Maren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Runa fully understood. The wooden shelves bowed slightly under the weight.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Reports from the harbour mentioned a sugar glider had been observed twice that week.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe workshop on Joelle Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Among the items catalogued was a moth, noted without further comment.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Nalini recalled that a eagle had appeared briefly near the market.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Among the items catalogued was a finch, noted without further comment.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Zora fully understood. The wooden shelves bowed slightly under the weight.\n\nThe workshop on Nalini Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The logbook recorded a butterfly at the northern edge of the district.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n---\n\nList ALL birds mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"dove\", \"osprey\", \"quail\", \"magpie\", \"raven\", \"toucan\", \"pelican\", \"parrot\", \"eagle\", \"finch\"], \"nearmisses\": [\"wasp\", \"flying squirrel\", \"flying fish\", \"dragonfly\", \"pterodactyl\", \"bat\", \"beetle\", \"sugar glider\", \"moth\", \"butterfly\"]}"
 }
]
''')

print(f"Loaded {len(DATASET)} items")
for tt in ['sustained']:
    count = sum(1 for d in DATASET if d["task_type"] == tt)
    print(f"  {tt}: {count} items")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 4: Execution Loop
# ══════════════════════════════════════════════════════════════════════

TASK_DISPATCH = {
    "sustained": cogattention_sustained,
}

n_total = len(DATASET)
for i, item in enumerate(DATASET):
    task_fn = TASK_DISPATCH[item["task_type"]]
    print(f"[{i+1}/{n_total}] {item['task_id']} ({item['difficulty']})")
    task_fn.run(
        llm=kbench.llm,
        prompt=item["prompt"],
        gold_json=item["gold_json"],
        task_id=item["task_id"],
        difficulty=item["difficulty"],
    )

print(f"\nCompleted {n_total} items for Sustained Attention")
